# LLM Evaluator Variability Comparison: Ollama vs vllm

This notebook compares evaluator variability across 8 experimental conditions:

1. **Ollama Sequential (no seed)**: No batching, no seed control
2. **Ollama Sequential (with seed)**: No batching, seed=42
3. **Ollama Parallel (no seed)**: With batching, no seed control
4. **Ollama Parallel (with seed)**: With batching, seed=42
5. **vllm Sequential (deterministic)**: No batching, seed=42, temperature=0, deterministic mode
6. **vllm Batch (deterministic)**: WITH batching, seed=42, temperature=0
7. **vllm 70B Tensor Parallel**: 70B models with 2 GPUs, no quantization
8. **vllm 70B AWQ**: 70B models with AWQ 4-bit quantization, single GPU

## Research Questions
- Does setting a seed reduce variability?
- Does batching affect variability (Ollama vs vllm)?
- Is vllm truly deterministic (variability ≈ 0)?
- Does quantization (AWQ) affect variability compared to tensor parallel?
- Which models show the most/least variability?

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 11

In [ ]:
# Configuration
BASE_DIR = Path('./replicas/replicas_temp_NO_0')
OUTPUT_DIR = Path('./analysis_results_comparison')
OUTPUT_DIR.mkdir(exist_ok=True)

# Data directories for each condition
DATA_DIRS = {
    'ollama_seq_noseed': BASE_DIR / 'replicas',
    'ollama_seq_seed': BASE_DIR / 'replicas_seed',
    'ollama_par_noseed': BASE_DIR / 'replicas',
    'ollama_par_seed': BASE_DIR / 'replicas_seed',
    'vllm_seq_seed': BASE_DIR / 'replicas_vllm_deterministic',
    'vllm_batch_seed': BASE_DIR / 'replicas_vllm_deterministic',  # batch_vllm files
    'vllm_70b_tp': BASE_DIR / 'replicas_vllm_deterministic',      # 70B with tensor parallel (no AWQ)
    'vllm_70b_awq': BASE_DIR / 'replicas_vllm_deterministic'      # 70B with AWQ quantization
}

# Model list (using common model names)
# Map vllm model names to ollama equivalents
VLLM_TO_OLLAMA = {
    'llama3.1:7b': 'llama3.1:latest',
    'qwen2.5:7b': 'qwen2.5:7b',
    'phi4:14b': 'phi4:latest',          # vllm uses phi4:14b, ollama uses phi4:latest
    'deepseek-r1:7b': 'deepseek-r1:7b',
    'llama3.3:70b': 'llama3.3:latest',
    'deepseek-r1:70b': 'deepseek-r1:70b'
}

# Models that are 70B (to handle AWQ and tensor parallel separately)
MODELS_70B = ['llama3.3:latest', 'deepseek-r1:70b']

MODELS = list(VLLM_TO_OLLAMA.values())
N_REPLICAS = 5

print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Models to analyze: {MODELS}")
print(f"70B models: {MODELS_70B}")
print(f"Number of replicas: {N_REPLICAS}")
print(f"\nConditions:")
for cond, path in DATA_DIRS.items():
    print(f"  {cond}: {path}")

## Load Data from All Conditions

In [ ]:
def load_condition_results(data_dir, condition_name, model_list, mode, n_replicas=5):
    """
    Load results for a specific condition.
    
    Args:
        data_dir: Directory containing CSV files
        condition_name: Name of the condition (e.g., 'ollama_seq_noseed', 'vllm_batch_seed')
        model_list: List of model names
        mode: 'sequential' or 'parallel'
        n_replicas: Number of replicas (default 5)
    
    Returns:
        Dictionary with DataFrames for each model
    """
    results = {}
    
    for model_name in model_list:
        model_safe = model_name.replace(':', '_')
        
        # Determine file pattern based on condition
        if 'vllm' in condition_name:
            # vllm: Load individual replica files and combine them
            # Map ollama model name back to vllm name
            vllm_model = [k for k, v in VLLM_TO_OLLAMA.items() if v == model_name]
            if not vllm_model:
                continue
            vllm_model_safe = vllm_model[0].replace(':', '_')
            
            # Determine file prefix and model filtering based on condition
            if condition_name == 'vllm_70b_tp':
                # Only 70B models WITHOUT AWQ (tensor parallel)
                if model_name not in MODELS_70B:
                    continue
                file_prefix = "sequential_vllm"
            elif condition_name == 'vllm_70b_awq':
                # Only 70B models WITH AWQ
                if model_name not in MODELS_70B:
                    continue
                file_prefix = "sequential_vllm_awq"
            elif 'batch' in condition_name:
                # Batch processing (all models)
                file_prefix = "batch_vllm"
            else:
                # Sequential (all models)
                file_prefix = "sequential_vllm"
            
            # Load individual replica files
            replica_dfs = []
            for replica_id in range(n_replicas):
                file_path = data_dir / f"{file_prefix}_{vllm_model_safe}_replica{replica_id}.csv"
                if file_path.exists():
                    df_replica = pd.read_csv(file_path)
                    df_replica['replica_id'] = replica_id
                    replica_dfs.append(df_replica)
                else:
                    print(f"  ⚠ Missing replica file: {file_path.name}")
            
            if replica_dfs:
                # Combine all replicas
                df = pd.concat(replica_dfs, ignore_index=True)
                df['condition'] = condition_name
                df['model'] = model_name
                results[model_name] = df
                print(f"✓ Loaded {condition_name} / {model_name}: {len(df)} evaluations ({len(replica_dfs)} replicas)")
            else:
                print(f"✗ No replica files found for {model_name}")
        else:
            # Ollama naming: use the combined file
            file_path = data_dir / f"{mode}_{model_safe}_all_replicas.csv"
            
            if file_path.exists():
                df = pd.read_csv(file_path)
                df['condition'] = condition_name
                df['model'] = model_name
                results[model_name] = df
                print(f"✓ Loaded {condition_name} / {model_name}: {len(df)} evaluations")
            else:
                print(f"✗ File not found: {file_path}")
    
    return results

# Load all conditions
all_data = {}

print("\n" + "="*80)
print("LOADING OLLAMA SEQUENTIAL (NO SEED)")
print("="*80)
all_data['ollama_seq_noseed'] = load_condition_results(
    DATA_DIRS['ollama_seq_noseed'], 'ollama_seq_noseed', MODELS, 'sequential'
)

print("\n" + "="*80)
print("LOADING OLLAMA SEQUENTIAL (WITH SEED)")
print("="*80)
all_data['ollama_seq_seed'] = load_condition_results(
    DATA_DIRS['ollama_seq_seed'], 'ollama_seq_seed', MODELS, 'sequential'
)

print("\n" + "="*80)
print("LOADING OLLAMA PARALLEL (NO SEED)")
print("="*80)
all_data['ollama_par_noseed'] = load_condition_results(
    DATA_DIRS['ollama_par_noseed'], 'ollama_par_noseed', MODELS, 'parallel'
)

print("\n" + "="*80)
print("LOADING OLLAMA PARALLEL (WITH SEED)")
print("="*80)
all_data['ollama_par_seed'] = load_condition_results(
    DATA_DIRS['ollama_par_seed'], 'ollama_par_seed', MODELS, 'parallel'
)

print("\n" + "="*80)
print("LOADING VLLM SEQUENTIAL (DETERMINISTIC, SEED=42)")
print("="*80)
all_data['vllm_seq_seed'] = load_condition_results(
    DATA_DIRS['vllm_seq_seed'], 'vllm_seq_seed', MODELS, 'sequential', n_replicas=N_REPLICAS
)

print("\n" + "="*80)
print("LOADING VLLM BATCH (DETERMINISTIC, SEED=42)")
print("="*80)
all_data['vllm_batch_seed'] = load_condition_results(
    DATA_DIRS['vllm_batch_seed'], 'vllm_batch_seed', MODELS, 'batch', n_replicas=N_REPLICAS
)

print("\n" + "="*80)
print("LOADING VLLM 70B TENSOR PARALLEL (NO AWQ)")
print("="*80)
all_data['vllm_70b_tp'] = load_condition_results(
    DATA_DIRS['vllm_70b_tp'], 'vllm_70b_tp', MODELS, 'sequential', n_replicas=N_REPLICAS
)

print("\n" + "="*80)
print("LOADING VLLM 70B AWQ QUANTIZED")
print("="*80)
all_data['vllm_70b_awq'] = load_condition_results(
    DATA_DIRS['vllm_70b_awq'], 'vllm_70b_awq', MODELS, 'sequential', n_replicas=N_REPLICAS
)

print("\n" + "="*80)
print("Data loading complete")
print("="*80)

## Calculate Variability Metrics for All Conditions

In [ ]:
def calculate_variability_metrics(df, condition, model):
    """
    Calculate variability metrics for a given condition and model.
    
    Args:
        df: DataFrame with evaluation results
        condition: Condition name
        model: Model name
    
    Returns:
        DataFrame with variability metrics per answer
    """
    # Debug: Check if success column exists and its values
    if 'success' not in df.columns:
        print(f"  ⚠ Warning: 'success' column not found for {model} ({condition})")
        print(f"    Available columns: {df.columns.tolist()}")
        # Assume all are successful if column doesn't exist
        df_success = df.copy()
    else:
        # Filter successful evaluations
        df_success = df[df['success'] == True].copy()
        
        # Debug info
        total = len(df)
        successful = len(df_success)
        unique_answers = df['answer_id'].nunique()
        unique_answers_success = df_success['answer_id'].nunique() if len(df_success) > 0 else 0
        
        print(f"  📊 {model} ({condition}):")
        print(f"    Total evaluations: {total}")
        print(f"    Successful evaluations: {successful} ({100*successful/total:.1f}%)")
        print(f"    Unique answers (total): {unique_answers}")
        print(f"    Unique answers (with success): {unique_answers_success}")
        
        if successful == 0:
            print(f"  ⚠ Warning: No successful evaluations found for {model} ({condition})")
            print(f"    Success value counts: {df['success'].value_counts().to_dict()}")
            return pd.DataFrame()
    
    if len(df_success) == 0:
        print(f"  ⚠ Warning: No data to process for {model} ({condition})")
        return pd.DataFrame()
    
    # Calculate variability per answer
    variability_data = []
    
    # Track answers by number of replicas
    replica_counts = {}
    
    for answer_id in df_success['answer_id'].unique():
        answer_df = df_success[df_success['answer_id'] == answer_id]
        n_replicas = len(answer_df)
        
        # Count by replica number
        replica_counts[n_replicas] = replica_counts.get(n_replicas, 0) + 1
        
        if n_replicas < 2:
            continue
        
        # Calculate standard deviation
        accuracy_std = answer_df['accuracy_score'].std()
        clarity_std = answer_df['clarity_score'].std()
        completeness_std = answer_df['completeness_score'].std()
        
        # Calculate range
        accuracy_range = answer_df['accuracy_score'].max() - answer_df['accuracy_score'].min()
        clarity_range = answer_df['clarity_score'].max() - answer_df['clarity_score'].min()
        completeness_range = answer_df['completeness_score'].max() - answer_df['completeness_score'].min()
        
        # Calculate coefficient of variation (CV)
        accuracy_mean = answer_df['accuracy_score'].mean()
        clarity_mean = answer_df['clarity_score'].mean()
        completeness_mean = answer_df['completeness_score'].mean()
        
        accuracy_cv = (accuracy_std / accuracy_mean) * 100 if accuracy_mean > 0 else 0
        clarity_cv = (clarity_std / clarity_mean) * 100 if clarity_mean > 0 else 0
        completeness_cv = (completeness_std / completeness_mean) * 100 if completeness_mean > 0 else 0
        
        variability_data.append({
            'answer_id': answer_id,
            'condition': condition,
            'model': model,
            'n_replicas': n_replicas,
            'accuracy_std': accuracy_std,
            'clarity_std': clarity_std,
            'completeness_std': completeness_std,
            'accuracy_range': accuracy_range,
            'clarity_range': clarity_range,
            'completeness_range': completeness_range,
            'accuracy_cv': accuracy_cv,
            'clarity_cv': clarity_cv,
            'completeness_cv': completeness_cv,
            'accuracy_mean': accuracy_mean,
            'clarity_mean': clarity_mean,
            'completeness_mean': completeness_mean
        })
    
    # Print replica distribution
    print(f"    Replica distribution:")
    for n_rep in sorted(replica_counts.keys()):
        print(f"      {n_rep} replicas: {replica_counts[n_rep]} answers")
    print(f"    ✓ Answers with ≥2 replicas (used for variability): {len(variability_data)}")
    
    return pd.DataFrame(variability_data)

# Calculate variability for all conditions and models
all_variability = []

for condition_name, condition_data in all_data.items():
    print(f"\n{'='*80}")
    print(f"Processing {condition_name}...")
    print(f"{'='*80}")
    for model_name, df in condition_data.items():
        if df is not None and len(df) > 0:
            var_df = calculate_variability_metrics(df, condition_name, model_name)
            if len(var_df) > 0:
                all_variability.append(var_df)

# Combine all variability data
df_variability = pd.concat(all_variability, ignore_index=True)

# Save variability metrics
output_file = OUTPUT_DIR / "variability_metrics_all_conditions.csv"
df_variability.to_csv(output_file, index=False)
print(f"\n{'='*80}")
print(f"✓ Variability metrics saved: {output_file}")
print(f"Total answers analyzed: {len(df_variability)}")
print(f"{'='*80}")

# Display summary
print(f"\n{'='*80}")
print("Variability Metrics Summary (Mean Std Dev)")
print(f"{'='*80}")
summary = df_variability.groupby(['condition', 'model'])[['accuracy_std', 'clarity_std', 'completeness_std']].mean()
print(summary.round(3))

## Summary Statistics by Condition

In [ ]:
# Overall summary by condition (averaged across all models)
print("\n" + "="*80)
print("OVERALL VARIABILITY BY CONDITION (All models averaged)")
print("="*80)

condition_summary = df_variability.groupby('condition').agg({
    'accuracy_std': ['mean', 'median', 'std'],
    'clarity_std': ['mean', 'median', 'std'],
    'completeness_std': ['mean', 'median', 'std']
}).round(4)

print(condition_summary)

# Save summary
summary_file = OUTPUT_DIR / "summary_by_condition.csv"
condition_summary.to_csv(summary_file)
print(f"\n✓ Summary saved: {summary_file}")

# Check for zero variability (perfect determinism)
print("\n" + "="*80)
print("ZERO VARIABILITY CHECK (Perfect Determinism)")
print("="*80)

for condition in df_variability['condition'].unique():
    cond_data = df_variability[df_variability['condition'] == condition]
    
    zero_accuracy = (cond_data['accuracy_std'] == 0).sum()
    zero_clarity = (cond_data['clarity_std'] == 0).sum()
    zero_completeness = (cond_data['completeness_std'] == 0).sum()
    total = len(cond_data)
    
    print(f"\n{condition}:")
    print(f"  Accuracy: {zero_accuracy}/{total} ({100*zero_accuracy/total:.1f}%) perfect replicas")
    print(f"  Clarity: {zero_clarity}/{total} ({100*zero_clarity/total:.1f}%) perfect replicas")
    print(f"  Completeness: {zero_completeness}/{total} ({100*zero_completeness/total:.1f}%) perfect replicas")

## Visualization 1: Variability Comparison Across Conditions

In [ ]:
# Plot 1: Box plots comparing variability across conditions
# For 70B models in vllm_seq_seed, use vllm_70b_awq data instead

# Create modified dataframe for plotting
df_variability_plot = df_variability.copy()


          

fig, axes = plt.subplots(1, 3, figsize=(26, 6))
metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
titles = ['Accuracy Variability (Std Dev)', 'Clarity Variability (Std Dev)', 'Completeness Variability (Std Dev)']

# Define colors for conditions
condition_colors = {
    'ollama_seq_noseed': '#e74c3c',    # Red
    'ollama_seq_seed': '#f39c12',      # Orange
    'ollama_par_noseed': '#3498db',    # Blue
    'ollama_par_seed': '#9b59b6',      # Purple
    'vllm_seq_seed': '#2ecc71',        # Green (70B will use AWQ data)
    'vllm_batch_seed': '#1abc9c'       # Teal
}

for ax, metric, title in zip(axes, metrics, titles):
    # Order conditions
    condition_order = ['ollama_seq_noseed', 'ollama_seq_seed', 'ollama_par_noseed', 
                       'ollama_par_seed', 'vllm_seq_seed', 'vllm_batch_seed']
    
    # Filter to only conditions that exist in the data
    available_conditions = [c for c in condition_order if c in df_variability_plot['condition'].unique()]
    
    # Prepare data in the correct order
    plot_data = []
    plot_colors = []
    for cond in available_conditions:
        plot_data.append(df_variability_plot[df_variability_plot['condition'] == cond][metric])
        plot_colors.append(condition_colors[cond])
    
    # Create boxplot
    bp = ax.boxplot(
        plot_data,
        labels=available_conditions,
        patch_artist=True,
        showfliers=False,  # Don't show outliers as we'll plot all points
        widths=0.6
    )
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], plot_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)  # More transparent to see points better
    
    # Add all individual points on top
    for i, cond in enumerate(available_conditions):
        y_data = df_variability_plot[df_variability_plot['condition'] == cond][metric]
        # Add jitter to x-coordinates to avoid overlapping points
        x_data = np.random.normal(i + 1, 0.04, size=len(y_data))
        ax.scatter(
            x_data, 
            y_data, 
            alpha=0.6, 
            s=30,
            color=condition_colors[cond],
            edgecolors='black',
            linewidths=0.5,
            zorder=3  # Draw points on top
        )
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Condition', fontsize=12)
    ax.set_ylabel('Standard Deviation', fontsize=12)
    ax.set_xticklabels([
        'Ollama\nSeq\nNo Seed',
        'Ollama\nSeq\nSeed',
        'Ollama\nPar\nNo Seed',
        'Ollama\nPar\nSeed',
        'vllm Seq\nSeed=42\n(70B:AWQ)',
        'vllm Batch\nSeed=42'
    ][:len(available_conditions)], rotation=0, ha='center', fontsize=10)
    ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
output_file = OUTPUT_DIR / "figure_variability_comparison_boxplot.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

In [ ]:
# Plot: Accuracy variability by condition for each model
# Two rows of three subplots (2x3 grid)

# Create modified dataframe for plotting
df_variability_plot = df_variability.copy()


# Get list of models
models = sorted(df_variability_plot['model'].unique())
n_models = len(models)

# Create subplots (2 rows, 3 columns)
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()  # Flatten to 1D array for easier iteration

# Define colors for conditions (paired colors for seq/batch)
condition_colors = {
    'ollama_seq_noseed': '#e74c3c',    # Red (API Seq No Seed)
    'ollama_par_noseed': '#c0392b',    # Dark Red (API Batch No Seed)
    'ollama_seq_seed': '#f39c12',      # Orange (API Seq Seed)
    'ollama_par_seed': '#d68910',      # Dark Orange (API Batch Seed)
    'vllm_seq_seed': '#2ecc71',        # Green (LOCAL Seq Seed - 70B will use AWQ data)
    'vllm_batch_seed': '#27ae60'       # Dark Green (LOCAL Batch Seed)
}

# Order conditions: Sequential vs Batch for each mode
condition_order = ['ollama_seq_noseed', 'ollama_par_noseed',   # API No Seed
                   'ollama_seq_seed', 'ollama_par_seed',       # API Seed
                   'vllm_seq_seed', 'vllm_batch_seed']         # LOCAL Seed

# Condition labels (shorter for x-axis)
condition_labels_short = {
    'ollama_seq_noseed': 'API\nSeq\nNo Seed',
    'ollama_par_noseed': 'API\nBatch\nNo Seed',
    'ollama_seq_seed': 'API\nSeq\nSeed',
    'ollama_par_seed': 'API\nBatch\nSeed',
    'vllm_seq_seed': 'LOCAL\nSeq\nSeed',
    'vllm_batch_seed': 'LOCAL\nBatch\nSeed'
}

# Plot each model
for idx, model in enumerate(models):
    if idx >= len(axes):
        break
    
    ax = axes[idx]
    model_data = df_variability_plot[df_variability_plot['model'] == model]
    
    # Get available conditions for this model
    available_conditions = [c for c in condition_order if c in model_data['condition'].unique()]
    
    # Prepare data
    plot_data = []
    plot_colors = []
    plot_means = []
    for cond in available_conditions:
        cond_data = model_data[model_data['condition'] == cond]['accuracy_std']
        plot_data.append(cond_data)
        plot_colors.append(condition_colors[cond])
        plot_means.append(cond_data.mean())
    
    # Create boxplot
    bp = ax.boxplot(
        plot_data,
        labels=available_conditions,
        patch_artist=True,
        showfliers=False,
        widths=0.6
    )
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], plot_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    
    # Add all individual points on top
    for i, cond in enumerate(available_conditions):
        y_data = model_data[model_data['condition'] == cond]['accuracy_std']
        # Add jitter to x-coordinates
        x_data = np.random.normal(i + 1, 0.04, size=len(y_data))
        ax.scatter(
            x_data, 
            y_data, 
            alpha=0.6, 
            s=40,
            color=condition_colors[cond],
            edgecolors='black',
            linewidths=0.5,
            zorder=3
        )
    
    # Add percentage of non-zero variability on top of each boxplot
    y_max = ax.get_ylim()[1]
    for i, cond in enumerate(available_conditions):
        cond_data = model_data[model_data['condition'] == cond]
        
        # Calculate percentage of responses with variability (std > 0)
        std_vals = cond_data['accuracy_std']
        non_zero_count = (std_vals > 0).sum()
        total_count = len(std_vals)
        percentage_non_zero = (non_zero_count / total_count * 100) if total_count > 0 else 0
        
        # Position text above the highest point
        text_y = max(std_vals) + 0.03 * y_max
        ax.text(
            i + 1, 
            text_y,
            f'{percentage_non_zero:.1f}%',
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold',
            color=condition_colors[cond],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='none', alpha=0.7)
        )
        
    # Formatting
    ax.set_title(f'{model}', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Condition', fontsize=11)
    
    # Only add y-label to leftmost plots (column 0)
    if idx % 3 == 0:
        ax.set_ylabel('Accuracy Std Dev', fontsize=12, fontweight='bold')
    
    ax.set_xticklabels(
        [condition_labels_short[c] for c in available_conditions],
        rotation=0, 
        ha='center', 
        fontsize=9
    )
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.set_axisbelow(True)

# Hide unused subplots if less than 6 models
for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

# Add main title
fig.suptitle('Accuracy Variability by Model and Condition', 
             fontsize=16, fontweight='bold', y=0.995)

plt.tight_layout()
output_file = OUTPUT_DIR / "figure_accuracy_variability_by_model.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

## Visualization 2: Condition Comparison with Plotly (Interactive)

In [ ]:
# Plot: Accuracy variability by condition for each model
# Two rows of three subplots (2x3 grid)

# Create modified dataframe for plotting
df_variability_plot = df_variability.copy()

# For 70B models, use vllm_70b_awq data instead of vllm_seq_seed
# Instead of replacing, we filter and combine properly

# Step 1: Remove 70B models from vllm_seq_seed condition
df_variability_plot = df_variability[
    ~((df_variability['model'].isin(MODELS_70B)) & (df_variability['condition'] == 'vllm_seq_seed'))
].copy()

# Step 2: Add 70B AWQ data but label it as vllm_seq_seed
awq_70b_data = df_variability[
    (df_variability['model'].isin(MODELS_70B)) & 
    (df_variability['condition'] == 'vllm_70b_awq')
].copy()

# Change the condition label to vllm_seq_seed for these rows
awq_70b_data['condition'] = 'vllm_seq_seed'

# Step 3: Combine everything
df_variability_plot = pd.concat([df_variability_plot, awq_70b_data], ignore_index=True)


# Get list of models
models = sorted(df_variability_plot['model'].unique())
n_models = len(models)

# Create subplots (2 rows, 3 columns)
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()  # Flatten to 1D array for easier iteration

# Define colors for conditions (paired colors for seq/batch)
condition_colors = {
    'ollama_seq_noseed': '#e74c3c',    # Red (API Seq No Seed)
    'ollama_par_noseed': '#c0392b',    # Dark Red (API Batch No Seed)
    'ollama_seq_seed': '#f39c12',      # Orange (API Seq Seed)
    'ollama_par_seed': '#d68910',      # Dark Orange (API Batch Seed)
    'vllm_seq_seed': '#2ecc71',        # Green (LOCAL Seq Seed - 70B will use AWQ data)
    'vllm_batch_seed': '#27ae60'       # Dark Green (LOCAL Batch Seed)
}

# Order conditions: Sequential vs Batch for each mode
condition_order = ['ollama_seq_noseed', 'ollama_par_noseed',   # API No Seed
                   'ollama_seq_seed', 'ollama_par_seed',       # API Seed
                   'vllm_seq_seed', 'vllm_batch_seed']         # LOCAL Seed

# Condition labels (shorter for x-axis)
condition_labels_short = {
    'ollama_seq_noseed': 'API\nSeq\nNo Seed',
    'ollama_par_noseed': 'API\nBatch\nNo Seed',
    'ollama_seq_seed': 'API\nSeq\nSeed',
    'ollama_par_seed': 'API\nBatch\nSeed',
    'vllm_seq_seed': 'LOCAL\nSeq\nSeed',
    'vllm_batch_seed': 'LOCAL\nBatch\nSeed'
}

# Plot each model
for idx, model in enumerate(models):
    if idx >= len(axes):
        break
    
    ax = axes[idx]
    model_data = df_variability_plot[df_variability_plot['model'] == model]
    
    # Get available conditions for this model
    available_conditions = [c for c in condition_order if c in model_data['condition'].unique()]
    
    # Prepare data
    plot_data = []
    plot_colors = []
    plot_means = []
    for cond in available_conditions:
        cond_data = model_data[model_data['condition'] == cond]['accuracy_std']
        plot_data.append(cond_data)
        plot_colors.append(condition_colors[cond])
        plot_means.append(cond_data.mean())
    
    # Create boxplot
    bp = ax.boxplot(
        plot_data,
        labels=available_conditions,
        patch_artist=True,
        showfliers=False,
        widths=0.6
    )
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], plot_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    
    # Add all individual points on top
    for i, cond in enumerate(available_conditions):
        y_data = model_data[model_data['condition'] == cond]['accuracy_std']
        # Add jitter to x-coordinates
        x_data = np.random.normal(i + 1, 0.04, size=len(y_data))
        ax.scatter(
            x_data, 
            y_data, 
            alpha=0.6, 
            s=40,
            color=condition_colors[cond],
            edgecolors='black',
            linewidths=0.5,
            zorder=3
        )
    
    # Add percentage of non-zero variability and sample size on top of each boxplot
    y_max = ax.get_ylim()[1]
    for i, cond in enumerate(available_conditions):
        cond_data = model_data[model_data['condition'] == cond]
        
        # Calculate percentage of responses with variability (std > 0)
        std_vals = cond_data['accuracy_std']
        non_zero_count = (std_vals > 0).sum()
        total_count = len(std_vals)
        percentage_non_zero = (non_zero_count / total_count * 100) if total_count > 0 else 0
        
        # Position text above the highest point
        text_y = max(std_vals) + 0.03 * y_max
        ax.text(
            i + 1, 
            text_y,
            f'{percentage_non_zero:.1f}%',
            ha='center',
            va='bottom',
            fontsize=8,
            fontweight='bold',
            color=condition_colors[cond],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='none', alpha=0.7)
        )
        
    # Formatting
    ax.set_title(f'{model}', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Condition', fontsize=11)
    
    # Only add y-label to leftmost plots (column 0)
    if idx % 3 == 0:
        ax.set_ylabel('Accuracy Std Dev', fontsize=12, fontweight='bold')
    
    ax.set_xticklabels(
        [condition_labels_short[c] for c in available_conditions],
        rotation=0, 
        ha='center', 
        fontsize=9
    )
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.set_axisbelow(True)

# Hide unused subplots if less than 6 models
for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

# Add main title
fig.suptitle('Accuracy Variability by Model and Condition', 
             fontsize=16, fontweight='bold', y=0.995)

plt.tight_layout()
output_file = OUTPUT_DIR / "figure_accuracy_variability_by_model.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

## Visualization 3: Model-Specific Comparison

In [ ]:
# Plot 3: Heatmap showing variability by model and condition
fig, axes = plt.subplots(1, 3, figsize=(28, 8))
metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
titles = ['Accuracy Std Dev', 'Clarity Std Dev', 'Completeness Std Dev']

for ax, metric, title in zip(axes, metrics, titles):
    pivot_data = df_variability.pivot_table(
        values=metric,
        index='model',
        columns='condition',
        aggfunc='mean'
    )
    
    # Reorder columns
    condition_order = ['ollama_seq_noseed', 'ollama_seq_seed', 'ollama_par_noseed', 
                       'ollama_par_seed', 'vllm_seq_seed', 'vllm_batch_seed',
                       'vllm_70b_tp', 'vllm_70b_awq']
    
    # Only include conditions that exist
    condition_order = [c for c in condition_order if c in pivot_data.columns]
    pivot_data = pivot_data[condition_order]
    
    sns.heatmap(
        pivot_data,
        annot=True,
        fmt='.3f',
        cmap='YlOrRd',
        ax=ax,
        cbar_kws={'label': 'Mean Std Dev'},
        linewidths=0.5
    )
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Model' if ax == axes[0] else '', fontsize=12)
    ax.set_xticklabels([
        'Ollama\nSeq\nNo Seed',
        'Ollama\nSeq\nSeed',
        'Ollama\nPar\nNo Seed',
        'Ollama\nPar\nSeed',
        'vllm\nSeq\nSeed',
        'vllm\nBatch\nSeed',
        'vllm\n70B TP',
        'vllm\n70B AWQ'
    ][:len(condition_order)], rotation=0, ha='center', fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
output_file = OUTPUT_DIR / "figure_variability_heatmap_by_model.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

## Statistical Tests: Comparing Conditions

In [ ]:
from itertools import combinations

print("\n" + "="*80)
print("PAIRWISE STATISTICAL TESTS (Mann-Whitney U)")
print("="*80)

test_results = []

# Get all condition pairs
conditions = df_variability['condition'].unique()
condition_pairs = list(combinations(conditions, 2))

for metric in ['accuracy_std', 'clarity_std', 'completeness_std']:
    print(f"\n--- {metric.replace('_std', '').upper()} ---")
    
    for cond1, cond2 in condition_pairs:
        data1 = df_variability[df_variability['condition'] == cond1][metric].dropna()
        data2 = df_variability[df_variability['condition'] == cond2][metric].dropna()
        
        if len(data1) > 0 and len(data2) > 0:
            # Mann-Whitney U test
            statistic, p_value = mannwhitneyu(data1, data2, alternative='two-sided')
            
            # Effect size (Cohen's d)
            pooled_std = np.sqrt((data1.std()**2 + data2.std()**2) / 2)
            cohens_d = (data1.mean() - data2.mean()) / pooled_std if pooled_std > 0 else 0
            
            significant = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
            
            print(f"{cond1} vs {cond2}:")
            print(f"  Mean: {data1.mean():.4f} vs {data2.mean():.4f}")
            print(f"  p-value: {p_value:.4e} {significant}")
            print(f"  Cohen's d: {cohens_d:.3f}")
            
            test_results.append({
                'metric': metric,
                'condition_1': cond1,
                'condition_2': cond2,
                'mean_1': data1.mean(),
                'mean_2': data2.mean(),
                'p_value': p_value,
                'cohens_d': cohens_d,
                'significant': 'Yes' if p_value < 0.05 else 'No'
            })

# Save test results
df_tests = pd.DataFrame(test_results)
test_file = OUTPUT_DIR / "statistical_tests_pairwise.csv"
df_tests.to_csv(test_file, index=False)
print(f"\n✓ Statistical test results saved: {test_file}")

## Key Findings Summary

In [ ]:
from datetime import datetime

print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)

print(f"\nAnalysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total answers analyzed: {df_variability['answer_id'].nunique()}")
print(f"Total evaluations: {len(df_variability) * N_REPLICAS}")

# 1. Overall variability by condition
print("\n--- 1. Overall Variability (Mean Std Dev) by Condition ---")
overall = df_variability.groupby('condition')[['accuracy_std', 'clarity_std', 'completeness_std']].mean()
print(overall.round(4))

# 2. Determinism check for vllm
print("\n--- 2. Determinism Check (vllm should have ~0 variability) ---")
vllm_data = df_variability[df_variability['condition'] == 'vllm_seq_seed']
print(f"vllm Sequential (Deterministic):")
print(f"  Mean accuracy std: {vllm_data['accuracy_std'].mean():.6f}")
print(f"  Mean clarity std: {vllm_data['clarity_std'].mean():.6f}")
print(f"  Mean completeness std: {vllm_data['completeness_std'].mean():.6f}")
print(f"  Perfect replicas: {(vllm_data['accuracy_std'] == 0).sum()}/{len(vllm_data)} accuracy")

# 3. Seed effect in Ollama
print("\n--- 3. Effect of Seed in Ollama (Sequential) ---")
ollama_seq_noseed = df_variability[df_variability['condition'] == 'ollama_seq_noseed']['accuracy_std'].mean()
ollama_seq_seed = df_variability[df_variability['condition'] == 'ollama_seq_seed']['accuracy_std'].mean()
reduction = ((ollama_seq_noseed - ollama_seq_seed) / ollama_seq_noseed) * 100
print(f"No seed: {ollama_seq_noseed:.4f}")
print(f"With seed: {ollama_seq_seed:.4f}")
print(f"Variability reduction: {reduction:.1f}%")

# 4. Batching effect in Ollama
print("\n--- 4. Effect of Batching in Ollama (with seed) ---")
ollama_par_seed = df_variability[df_variability['condition'] == 'ollama_par_seed']['accuracy_std'].mean()
batching_effect = ((ollama_par_seed - ollama_seq_seed) / ollama_seq_seed) * 100
print(f"Sequential: {ollama_seq_seed:.4f}")
print(f"Parallel: {ollama_par_seed:.4f}")
print(f"Batching increases variability by: {batching_effect:.1f}%")

# 5. Most/least variable models
print("\n--- 5. Model Variability Ranking (across all conditions) ---")
model_var = df_variability.groupby('model')[['accuracy_std', 'clarity_std', 'completeness_std']].mean()
model_var['avg_variability'] = model_var.mean(axis=1)
model_var_sorted = model_var.sort_values('avg_variability')
print(f"Most stable: {model_var_sorted.index[0]} (avg std: {model_var_sorted['avg_variability'].iloc[0]:.4f})")
print(f"Least stable: {model_var_sorted.index[-1]} (avg std: {model_var_sorted['avg_variability'].iloc[-1]:.4f})")


print(f"All results saved in: {OUTPUT_DIR}")


In [ ]:
# Bar plot: Variability by model for each condition
# This shows how each model performs across all conditions

fig, axes = plt.subplots(1, 3, figsize=(26, 8))
metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
titles = ['Accuracy Variability (Std Dev)', 'Clarity Variability (Std Dev)', 'Completeness Variability (Std Dev)']

# Define colors for conditions
condition_colors = {
    'ollama_seq_noseed': '#e74c3c',    # Red
    'ollama_seq_seed': '#f39c12',      # Orange
    'ollama_par_noseed': '#3498db',    # Blue
    'ollama_par_seed': '#9b59b6',      # Purple
    'vllm_seq_seed': '#2ecc71',        # Green
    'vllm_batch_seed': '#1abc9c',      # Teal
    'vllm_70b_tp': '#27ae60',          # Dark green  
    'vllm_70b_awq': '#34495e'          # Dark gray
}

# Readable condition names
condition_labels = {
    'ollama_seq_noseed': 'Ollama Seq\nNo Seed',
    'ollama_seq_seed': 'Ollama Seq\nSeed=42',
    'ollama_par_noseed': 'Ollama Par\nNo Seed',
    'ollama_par_seed': 'Ollama Par\nSeed=42',
    'vllm_seq_seed': 'vllm Seq\nSeed=42',
    'vllm_batch_seed': 'vllm Batch\nSeed=42',
    'vllm_70b_tp': 'vllm 70B\nTensor Par',
    'vllm_70b_awq': 'vllm 70B\nAWQ'
}

# Prepare data: mean variability per model per condition
for ax, metric, title in zip(axes, metrics, titles):
    # Pivot data: models as rows, conditions as columns
    pivot_data = df_variability.pivot_table(
        values=metric,
        index='model',
        columns='condition',
        aggfunc='mean'
    )
    
    # Reorder conditions
    condition_order = ['ollama_seq_noseed', 'ollama_seq_seed', 'ollama_par_noseed', 
                       'ollama_par_seed', 'vllm_seq_seed', 'vllm_batch_seed',
                       'vllm_70b_tp', 'vllm_70b_awq']
    
    # Only include conditions that exist in the data
    condition_order = [c for c in condition_order if c in pivot_data.columns]
    pivot_data = pivot_data[condition_order]
    
    # Create grouped bar plot
    x = np.arange(len(pivot_data.index))  # Model positions
    width = 0.10  # Width of each bar (narrower with more conditions)
    
    for i, condition in enumerate(condition_order):
        offset = width * (i - len(condition_order) / 2 + 0.5)
        bars = ax.bar(
            x + offset,
            pivot_data[condition],
            width,
            label=condition_labels.get(condition, condition),
            color=condition_colors.get(condition, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.5
        )
    
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('Mean Standard Deviation', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot_data.index, rotation=45, ha='right', fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    # Add legend only to the rightmost plot
    if ax == axes[-1]:
        ax.legend(
            loc='upper left',
            fontsize=9,
            framealpha=0.95,
            edgecolor='black',
            ncol=1
        )

plt.tight_layout()
output_file = OUTPUT_DIR / "figure_variability_by_model_and_condition.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

In [ ]:
# Bar plot: Variability by model for each condition
# This shows how each model performs across all conditions

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
titles = ['Accuracy Variability (Std Dev)', 'Clarity Variability (Std Dev)', 'Completeness Variability (Std Dev)']

# Prepare data for grouped bar plot
# We want: X-axis = models, grouped bars = conditions

# Define colors for conditions
condition_colors = {
    'ollama_seq_noseed': '#e74c3c',    # Red
    'ollama_seq_seed': '#f39c12',      # Orange
    'ollama_par_noseed': '#3498db',    # Blue
    'ollama_par_seed': '#9b59b6',      # Purple
    'vllm_seq_seed': '#2ecc71',        # Green
    'vllm_batch_seed': '#1abc9c',      # Teal
    'vllm_awq_seed': '#34495e'         # Dark gray
}

# Readable condition names
condition_labels = {
    'ollama_seq_noseed': 'Ollama Seq\nNo Seed',
    'ollama_seq_seed': 'Ollama Seq\nSeed=42',
    'ollama_par_noseed': 'Ollama Par\nNo Seed',
    'ollama_par_seed': 'Ollama Par\nSeed=42',
    'vllm_seq_seed': 'vllm Seq\nSeed=42',
    'vllm_batch_seed': 'vllm Batch\nSeed=42',
    'vllm_awq_seed': 'vllm AWQ\nSeed=42'
}

# Prepare data: mean variability per model per condition
for ax, metric, title in zip(axes, metrics, titles):
    # Pivot data: models as rows, conditions as columns
    pivot_data = df_variability.pivot_table(
        values=metric,
        index='model',
        columns='condition',
        aggfunc='mean'
    )
    
    # Reorder conditions
    condition_order = ['ollama_seq_noseed', 'ollama_seq_seed', 'ollama_par_noseed', 
                       'ollama_par_seed', 'vllm_seq_seed', 'vllm_batch_seed', 'vllm_awq_seed']
    
    # Only include conditions that exist in the data
    condition_order = [c for c in condition_order if c in pivot_data.columns]
    pivot_data = pivot_data[condition_order]
    
    # Create grouped bar plot
    x = np.arange(len(pivot_data.index))  # Model positions
    width = 0.12  # Width of each bar
    
    for i, condition in enumerate(condition_order):
        offset = width * (i - len(condition_order) / 2 + 0.5)
        bars = ax.bar(
            x + offset,
            pivot_data[condition],
            width,
            label=condition_labels.get(condition, condition),
            color=condition_colors.get(condition, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.5
        )
    
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('Mean Standard Deviation', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot_data.index, rotation=45, ha='right', fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    # Add legend only to the rightmost plot
    if ax == axes[-1]:
        ax.legend(
            loc='upper left',
            fontsize=10,
            framealpha=0.95,
            edgecolor='black',
            ncol=1
        )

plt.tight_layout()
output_file = OUTPUT_DIR / "figure_variability_by_model_and_condition.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

## Visualization 4: Variability by Model and Condition (Bar Plot)